# Gabor Filter Bank

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
import ipywidgets as widgets
from IPython.display import display

SAMPLE_IMG_DIR = Path.cwd() / "imgs"

image_paths = sorted(
    p for p in SAMPLE_IMG_DIR.iterdir()
    if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
)

image_picker = widgets.Dropdown(
    options=[(p.name, p) for p in image_paths],
    description="Image: ",
    layout=widgets.Layout(width="500px"),
)

display(image_picker)

In [ ]:
IMG_PATH: Optional[str] = "/Users/elij/Desktop/Imagery/di-img/afg.jpg"
height, width = 720, 960

In [ ]:
thetas: list[float] = list(np.linspace(0, np.pi, 6, endpoint=False))  # 0°, 30°, 60°, 90°, 120°, 150°
lambdas: list[float] = [6.0, 12.0, 24.0]  # pixels per cycle (small -> fine; large -> coarse)
gamma: float = 0.5  # spatial aspect ratio (ellipticity)
# sigma tied to lambda for ~1 octave bandwidth (rule of thumb); sigma ≈ 0.56*lambda
sigma_factor: float = 0.56

In [ ]:
def load_image_or_synthetic(path: Optional[str], H: int, W: int) -> np.ndarray:
    if path and os.path.exists(path):
        img = plt.imread(path)
        
        # if RGB(A) -> grayscale via per-pixel dot product (no '@')
        if img.ndim == 3:
            img = img[..., :3]  # drop alpha if present
            img = img.astype(np.float64, copy=False)
            RGB_TO_GRAY = np.array([0.2126, 0.7152, 0.0722], dtype=np.float64)
            img = np.tensordot(img, RGB_TO_GRAY, axes=([-1], [0]))  # shape (H, W)
        
        # safely normalize to [0, 1]
        img = img.astype(np.float64)
        
        if np.issubdtype(img.dtype, np.integer):
            maxv = np.iinfo(img.dtype).max
            img = img / maxv
        elif img.max() > 1.5:
            img = img / 255.0

        img = np.nan_to_num(img, nan=0.0, posinf=1.0, neginf=0.0)
        img_h, img_w = img.shape
        
        if img_h != H or img_w != W:
            # simple center crop/pad to HxW
            out = np.zeros((H, W), dtype=np.float64)
            out[:] = np.median(img)
            y0 = max(0, (H - img_h) // 2)
            x0 = max(0, (W - img_w) // 2)
            y1 = y0 + min(H, img_h)
            x1 = x0 + min(W, img_w)
            ys = max(0, (img_h - H) // 2)
            xs = max(0, (img_w - W) // 2)
            out[y0:y1, x0:x1] = img[ys:ys+(y1-y0), xs:xs+(x1-x0)]
            img = out
        return np.clip(img, 0.0, 1.0)

In [ ]:
def gabor_kernel(size: int, sigma: float, lam: float, theta: float, psi: float, gamma: float) -> np.ndarray:
    r = size // 2
    y, x = np.mgrid[-r:r+1, -r:r+1]
    # rotate coords
    xr = x*np.cos(theta) + y*np.sin(theta)
    yr = -x*np.sin(theta) + y*np.cos(theta)
    gauss = np.exp(-(xr**2 + (gamma**2)*(yr**2)) / (2*sigma*2))
    carrier = np.cos(2*np.pi*xr/lam + psi)
    g = gauss * carrier
    g -= g.mean()
    # normalize L2 to maintain comparable energy across sizes
    norm = np.linalg.norm(g.ravel()) + 1e-12
    return g / norm

In [ ]:
def fft_convolve2d(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    H, W = image.shape
    Kh, Kw = kernel.shape
    # full convolution via FFT
    outH, outW = H + Kh - 1, W + Kw - 1
    f_img = np.fft.rfft2(image, s=(outH, outW))
    # flip kernel for convolution (vs correlation)
    k_flip = np.flipud(np.fliplr(kernel))
    f_ker = np.fft.rfft2(k_flip, s=(outH, outW))
    conv_full = np.fft.irfft2(f_img * f_ker, s=(outH, outW))
    # extract the central region to match image size
    y0 = (Kh - 1) // 2
    x0 = (Kw - 1) // 2
    conv_same = conv_full[y0:y0+H, x0:x0+W]
    return conv_same


def normalize01(x: np.ndarray) -> np.ndarray:
    x = x - x.min()
    d = x.max() - x.min()
    return x / (d + 1e-12)

In [ ]:
# build image
img = load_image_or_synthetic(IMG_PATH, height, width)

# compute responses
responses: dict[tuple[float, float], np.ndarray] = {}
energies: dict[tuple[float, float], float] = {}

for lam in lambdas:
    sigma = sigma_factor * lam
    # kernel size ~ 8*sigma (odd)
    ksize = int(np.ceil(8.0 * sigma))
    if ksize % 2 == 0:
        ksize += 1

    for th in thetas:
        # Quadrature pair: psi=0 (even), psi=pi/2 (odd) -> amplitude envelope
        g_even = gabor_kernel(ksize, sigma, lam, th, psi=0.0, gamma=gamma)
        g_odd = gabor_kernel(ksize, sigma, lam, th, psi=np.pi/2, gamma=gamma)
        r_even = fft_convolve2d(img, g_even)
        r_odd = fft_convolve2d(img, g_odd)
        amp = np.sqrt(r_even**2 + r_odd**2)  # phase-insensitive energy
        responses[(lam, th)] = amp
        energies[(lam, th)] = float(np.sum(amp))

In [ ]:
# build bank mosaic image (3x6 grid)
tile_h, tile_w = img.shape
GAP = 2  # gutter between tiles
rows = len(lambdas)
cols = len(thetas)
mosaic_h = rows*tile_h + (rows-1)*GAP
mosaic_w = cols*tile_w + (cols-1)*GAP
mosaic = np.ones((mosaic_h, mosaic_w), dtype=np.float64) * 0.0  # black background

for i, lam in enumerate(lambdas):
    for j, th in enumerate(thetas):
        tile = normalize01(responses[(lam, th)])
        y0 = i*(tile_h + GAP)
        x0 = j*(tile_w + GAP)
        mosaic[y0:y0+tile_h, x0:x0+tile_w] = tile